In [1]:
%pip install -U langchain langchain-core langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [9]:

import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6I3QZYxBztg-W1wpiuKsFYFfHX3gHcnmJfbjWCvsgAo7Q"

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{question}")
])

chain = prompt | llm

products = pd.DataFrame({
    "product": ["Laptop A", "Laptop B", "Smartphone A", "Smartphone B", "Headphones A"],
    "price": [85000, 120000, 65000, 95000, 15000],
    "currency": ["PKR", "PKR", "PKR", "PKR", "PKR"]
})

products.to_csv("products.csv", index=False)

@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Perform addition, subtraction, multiplication, or division on two numbers."""
    
    operation = operation.strip().lower()

    if operation in ["add", "addition", "plus"]:
        return a + b

    elif operation in ["subtract", "subtraction", "minus"]:
        return a - b

    elif operation in ["multiply", "multiplication", "times"]:
        return a * b

    elif operation in ["divide", "division"]:
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b

    else:
        raise ValueError(
            f"Unknown operation: {operation}. "
            "Use add, subtract, multiply, or divide."
        )

print("Calculator tool updated successfully.")
@tool
def get_weather(city: str) -> dict:
    """Return weather information including temperature and condition for a supported city."""
    weather_data = {
        "karachi": {"temperature": 34, "condition": "Sunny"},
        "lahore": {"temperature": 31, "condition": "Partly cloudy"},
        "islamabad": {"temperature": 27, "condition": "Cloudy"},
        "dubai": {"temperature": 38, "condition": "Sunny"}
    }
    city_key = city.strip().lower()
    if city_key not in weather_data:
        raise ValueError(f"Weather data not available for {city}.")
    return weather_data[city_key]

@tool
def get_product_price(product_name: str) -> dict:
    """Read a product price from products.csv and return its price and currency."""
    data = pd.read_csv("products.csv")
    match = data[data["product"].str.lower() == product_name.strip().lower()]
    if match.empty:
        raise ValueError(f"Product not found: {product_name}")
    row = match.iloc[0]
    return {
        "product": row["product"],
        "price": float(row["price"]),
        "currency": row["currency"]
    }

tools = [calculator, get_weather, get_product_price]

print("Task 1 + Task 2 setup completed successfully.")
print("LLM:", type(llm).__name__)
print("Tools:", [tool.name for tool in tools])

print("\nCalculator Test:")
print(calculator.invoke({"a": 20, "b": 5, "operation": "multiply"}))

print("\nWeather Test:")
print(get_weather.invoke({"city": "Karachi"}))

print("\nProduct Price Test:")
print(get_product_price.invoke({"product_name": "Laptop A"}))

Calculator tool updated successfully.
Task 1 + Task 2 setup completed successfully.
LLM: ChatGoogleGenerativeAI
Tools: ['calculator', 'get_weather', 'get_product_price']

Calculator Test:
100.0

Weather Test:
{'temperature': 34, 'condition': 'Sunny'}

Product Price Test:
{'product': 'Laptop A', 'price': 85000.0, 'currency': 'PKR'}


In [4]:
import langchain

print("LangChain version:", langchain.__version__)

LangChain version: 1.4.0


In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Use the available tools whenever needed. "
        "Use calculator for calculations, "
        "get_weather for weather, and "
        "get_product_price for product prices."
    )
)

print("Task 3 agent created successfully!")

Task 3 agent created successfully!


In [11]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of Laptop A?"
        }
    ]
})

print(result)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'messages': [HumanMessage(content='What is the price of Laptop A?', additional_kwargs={}, response_metadata={}, id='8dcc90ea-5188-4a12-9cf4-ed3ada6c688b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_product_price', 'arguments': '{"product_name": "Laptop A"}'}, '__gemini_function_call_thought_signatures__': {'call_314726': 'El4KXAERTTIPJ2XjBmgIhfr0/yB6VzvBXXu2uOhHOK5e6jDZCib78cluu91rxETdrhMd8RR1gJYY0uAB5PlOXKBajF3KSTcceJEJ/DL9haoch/qraFFPtxWJwdiW1IMv'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a080c7-6374-7a03-97bf-d4c7972995e1-0', tool_calls=[{'name': 'get_product_price', 'args': {'product_name': 'Laptop A'}, 'id': 'call_314726', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 21, 'total_tokens': 244, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='{"product": "Laptop A", "pri

In [12]:
final_answer = result["messages"][-1].content

if isinstance(final_answer, list):
    final_answer = final_answer[0]["text"]

print("FINAL ANSWER:")
print(final_answer)

FINAL ANSWER:
The price of Laptop A is 85,000 PKR.


In [13]:
result2 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Laptop A costs 85000 PKR. Calculate its price after a 10% increase."
        }
    ]
})

final_answer2 = result2["messages"][-1].content

if isinstance(final_answer2, list):
    final_answer2 = final_answer2[0]["text"]

print("FINAL ANSWER:")
print(final_answer2)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


FINAL ANSWER:
The price of Laptop A after a 10% increase is **93,500 PKR**.


In [14]:
result2 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Laptop A costs 85000 PKR. Calculate its price after a 10% increase."
        }
    ]
})

final_answer2 = result2["messages"][-1].content

if isinstance(final_answer2, list):
    final_answer2 = final_answer2[0]["text"]

print("FINAL ANSWER:")
print(final_answer2)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


FINAL ANSWER:
The price of Laptop A after a 10% increase is **93,500 PKR**.


In [15]:
print("TASK 3 TRACE")
print("-" * 50)

for message in result2["messages"]:
    print(type(message).__name__, ":", message.content)

TASK 3 TRACE
--------------------------------------------------
HumanMessage : Laptop A costs 85000 PKR. Calculate its price after a 10% increase.
AIMessage : []
ToolMessage : 93500.0
AIMessage : [{'type': 'text', 'text': 'The price of Laptop A after a 10% increase is **93,500 PKR**.', 'extras': {'signature': 'El4KXAERTTIPr+5LC9lEQQ81z7eTSgW4l0d+7gWI8KuxOG6UFLz0EUmndsZoQqtCDjUDXTKS4tNfKXTANt5oCO3yuL/6v417e4YJpiy+VeWT0d0fSm0ZsTxMN2FwSfV9'}}]


# Task 4: Conversation Memory

In this task, conversation memory is added to the LangChain agent so that it can remember information from previous turns and use that information in follow-up questions.

The memory system stores the conversation history for each session and sends the previous messages along with new user input.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

memory_store = {}

def get_history(session_id):
    if session_id not in memory_store:
        memory_store[session_id] = []
    return memory_store[session_id]

def run_with_memory(session_id, user_input):
    history = get_history(session_id)

    messages = history + [
        HumanMessage(content=user_input)
    ]

    result = agent.invoke({
        "messages": messages
    })

    response = result["messages"][-1].content

    if isinstance(response, list):
        response = response[0]["text"]

    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))

    return response


Task 4 memory system created successfully!


In [17]:
session_id = "client_001"

response1 = run_with_memory(
    session_id,
    "Laptop A costs 85000 PKR."
)

print("TURN 1:")
print(response1)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TURN 1:
Got it! Laptop A is priced at 85,000 PKR. Let me know if you need any help comparing it with other laptops, calculating something, or checking product prices!


In [18]:
response2 = run_with_memory(
    session_id,
    "What is its price after a 10% increase?"
)

print("TURN 2:")
print(response2)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TURN 2:
The price of Laptop A after a 10% increase is **93,500 PKR**.


In [19]:
response3 = run_with_memory(
    session_id,
    "Would you recommend buying it?"
)

print("TURN 3:")
print(response3)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TURN 3:
Whether or not I would recommend buying **Laptop A** (at 85,000 PKR, or 93,500 PKR after the increase) depends heavily on what you plan to use it for and its exact specifications! 

To give you the best advice, could you share a bit more detail?
1. **What are the specifications?** (Processor, RAM, Storage, Graphics card, Screen size/resolution)
2. **What will you be using it for?** (Gaming, video editing, software development, university/office work, or just casual browsing?)
3. **What are the alternatives?** Are you comparing it with other laptops in a similar price range?

Generally speaking, in the 85k–95k PKR range in Pakistan:
* If it has an **i5 or Ryzen 5 processor, 8GB/16GB RAM, and an SSD**, it’s usually a great daily driver for students and professionals.
* If it’s for **heavy gaming or professional 3D rendering**, you might need something with a dedicated graphics card (like an NVIDIA GTX/RTX), which typically costs more.

Let me know the specs, and I'll tell you if 

In [20]:
print("CONVERSATION MEMORY")
print("-" * 50)

for message in memory_store[session_id]:
    print(type(message).__name__, ":", message.content)

CONVERSATION MEMORY
--------------------------------------------------
HumanMessage : Laptop A costs 85000 PKR.
AIMessage : Got it! Laptop A is priced at 85,000 PKR. Let me know if you need any help comparing it with other laptops, calculating something, or checking product prices!
HumanMessage : What is its price after a 10% increase?
AIMessage : The price of Laptop A after a 10% increase is **93,500 PKR**.
HumanMessage : Would you recommend buying it?
AIMessage : Whether or not I would recommend buying **Laptop A** (at 85,000 PKR, or 93,500 PKR after the increase) depends heavily on what you plan to use it for and its exact specifications! 

To give you the best advice, could you share a bit more detail?
1. **What are the specifications?** (Processor, RAM, Storage, Graphics card, Screen size/resolution)
2. **What will you be using it for?** (Gaming, video editing, software development, university/office work, or just casual browsing?)
3. **What are the alternatives?** Are you compari

In [22]:
print("TASK 4 - MEMORY VERIFICATION")

print("Number of stored messages:", len(memory_store[session_id]))

for i, message in enumerate(memory_store[session_id], 1):
    print(f"{i}. {type(message).__name__}: {message.content}")

TASK 4 - MEMORY VERIFICATION
Number of stored messages: 6
1. HumanMessage: Laptop A costs 85000 PKR.
2. AIMessage: Got it! Laptop A is priced at 85,000 PKR. Let me know if you need any help comparing it with other laptops, calculating something, or checking product prices!
3. HumanMessage: What is its price after a 10% increase?
4. AIMessage: The price of Laptop A after a 10% increase is **93,500 PKR**.
5. HumanMessage: Would you recommend buying it?
6. AIMessage: Whether or not I would recommend buying **Laptop A** (at 85,000 PKR, or 93,500 PKR after the increase) depends heavily on what you plan to use it for and its exact specifications! 

To give you the best advice, could you share a bit more detail?
1. **What are the specifications?** (Processor, RAM, Storage, Graphics card, Screen size/resolution)
2. **What will you be using it for?** (Gaming, video editing, software development, university/office work, or just casual browsing?)
3. **What are the alternatives?** Are you comparin

In [23]:
response4 = run_with_memory(
    session_id,
    "What was the original price of the laptop we discussed?"
)

print("MEMORY TEST:")
print(response4)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


MEMORY TEST:
The original price of Laptop A before the 10% increase was **85,000 PKR**.


## Memory Implementation

A session-based memory store is used to maintain conversation history. Each session has its own message history, allowing the agent to remember previous interactions.

The `run_with_memory()` function retrieves the previous conversation, adds the new user message, invokes the agent, and stores the new AI response back into memory.

## Three-Turn Conversation

The agent is tested using a three-turn conversation:

1. The user provides the price of Laptop A.
2. The user asks for the price after a 10% increase without repeating the laptop name.
3. The user asks whether the laptop should be recommended.

The second and third turns demonstrate that the agent can use information from previous conversation turns.


## Task 4 Conclusion

Conversation memory was successfully implemented using a session-based message history. The agent was able to remember the laptop price across multiple turns and correctly use previous information when answering follow-up questions. This makes the LangChain agent more conversational and useful than a stateless agent.

## Task 5: Structured Output and Error Handling

In this task, structured output is added to the LangChain agent using Pydantic. Tool failure handling is also tested to make the agent more reliable when invalid inputs or unsupported requests are provided.


In [24]:
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    product: str = Field(description="Product name")
    price: float = Field(description="Product price")
    currency: str = Field(description="Currency")
    recommendation: str = Field(description="Buying recommendation")
    reason: str = Field(description="Reason for the recommendation")

structured_llm = llm.with_structured_output(ProductRecommendation)

print("Structured output model created successfully!")

Structured output model created successfully!


In [25]:
structured_result = structured_llm.invoke(
    """
    Laptop A costs 85000 PKR.
    Based on this information, provide a buying recommendation.
    """
)

print("STRUCTURED OUTPUT:")
print(structured_result)

print("\nProduct:", structured_result.product)
print("Price:", structured_result.price)
print("Currency:", structured_result.currency)
print("Recommendation:", structured_result.recommendation)
print("Reason:", structured_result.reason)

c:\Users\samir\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


STRUCTURED OUTPUT:
product='Laptop A' price=85000.0 currency='PKR' recommendation='Consider purchasing' reason='It is a reasonable price for a standard laptop in the current market, provided the specifications meet your daily computing needs.'

Product: Laptop A
Price: 85000.0
Currency: PKR
Recommendation: Consider purchasing
Reason: It is a reasonable price for a standard laptop in the current market, provided the specifications meet your daily computing needs.


In [26]:
try:
    result = get_product_price.invoke({
        "product_name": "Laptop Z"
    })

    print("Result:")
    print(result)

except Exception as e:
    print("TOOL ERROR HANDLED:")
    print(e)

TOOL ERROR HANDLED:
Product not found: Laptop Z


In [27]:
try:
    result = get_weather.invoke({
        "city": "Multan"
    })

    print("Result:")
    print(result)

except Exception as e:
    print("WEATHER TOOL ERROR HANDLED:")
    print(e)

WEATHER TOOL ERROR HANDLED:
Weather data not available for Multan.


## Task 5 Conclusion

Structured output was successfully implemented using Pydantic, making the agent responses consistent and easy to process programmatically. Tool failure handling was also tested to prevent the application from crashing when invalid products or unsupported cities are requested. Compared with the raw-Python agent, LangChain provides a cleaner and more organized framework for tools, agent execution, memory, and structured responses. Overall, LangChain makes the agent easier to extend, maintain, and integrate into larger applications.
